In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.13.5 x86_64-unknown-linux-gnu
installing to /home/codespace/.local/bin
  uv
  uvx
everything's installed!


In [2]:
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [3]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


In [4]:
!git clone https://github.com/DataTalksClub/machine-learning-zoomcamp.git

Cloning into 'machine-learning-zoomcamp'...
remote: Enumerating objects: 18055, done.
remote: Counting objects: 100% (976/976), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 18055 (delta 925), reused 921 (delta 905), pack-reused 17079 (from 3)
Receiving objects: 100% (18055/18055), 996.12 MiB | 35.23 MiB/s, done.
Resolving deltas: 100% (11284/11284), done.
Updating files: 100% (1785/1785), done.


In [5]:
%cd machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

/workspaces/homework_05/homework_05/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment


In [6]:
!ls

Dockerfile	       homework.yaml	    predict.py	    train.py
README.md	       model.py		    pyproject.toml  uv.lock
feature_defaults.json  model_metadata.json  q6_test.py
homework.md	       pipeline.bin	    smoke_test.py


In [7]:
!uv sync --locked

Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 4ms
Prepared 18 packages in 908ms                                            
░░░░░░░░░░░░░░░░░░░░ [0/18] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 1.69s                              
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0


In [8]:
!uv sync --locked

Resolved 19 packages in 0.93ms
Checked 18 packages in 7ms


In [9]:
uv run python smoke_test.py

/usr/local/python/3.14.2/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [11]:
!~/.local/bin/uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [12]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


In [13]:
!grep -A 2 'name = "scikit-learn"' uv.lock

    { name = "scikit-learn" },
    { name = "uvicorn" },
]
--
    { name = "scikit-learn", specifier = "==1.7.2" },
    { name = "uvicorn", specifier = "==0.38.0" },
]
--
name = "scikit-learn"
version = "1.7.2"
source = { registry = "https://pypi.org/simple" }


In [14]:
%%writefile q3.py

import pickle

with open("pipeline.bin", "rb") as f:
    model = pickle.load(f)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

probability = model.predict_proba([lead])[0, 1]

print(probability)
print(round(probability, 3))

Writing q3.py


In [15]:
!~/.local/bin/uv run python q3.py

0.5329944186432507
0.533


In [16]:

import subprocess

server = subprocess.Popen([
    "/home/codespace/.local/bin/uv",
    "run",
    "uvicorn",
    "predict:app",
    "--host", "0.0.0.0",
    "--port", "9696"
])

INFO:     Started server process [7446]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)


In [17]:
!curl -s http://localhost:9696/health

INFO:     127.0.0.1:56128 - "GET /health HTTP/1.1" 200 OK
{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}

In [18]:
!curl -s http://localhost:9696/predict -H "Content-Type: application/json" -d '{"lead_source":"organic_search","industry":"technology","employment_status":"employed","location":"europe","number_of_courses_viewed":4,"annual_income":80304.0,"interaction_count":7,"lead_score":0.74}'

INFO:     127.0.0.1:47634 - "POST /predict HTTP/1.1" 200 OK
{"conversion_probability":0.769799,"conversion":true}

In [19]:
!head -n 5 Dockerfile

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \


In [20]:
server.terminate()

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [7446]


In [21]:
!docker build -t zoomcamp-model:2026-hw5 .

[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.2s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.2s
[+] Building 0.3s (1/4)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.2s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.2s
 => [auth] astral-sh/uv:pull token for ghcr.io                             0.0s
[+] Building 0.5s (3/5)                 

In [22]:
import subprocess

container = subprocess.Popen([
    "docker", "run",
    "--rm",
    "-p", "9696:9696",
    "zoomcamp-model:2026-hw5"
])

In [23]:
!curl -s http://localhost:9696/health

INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)


In [24]:
!~/.local/bin/uv run python q6_test.py

INFO:     172.17.0.1:51060 - "POST /predict HTTP/1.1" 200 OK
{'conversion_probability': 0.769799, 'conversion': True}
